# OpenCode Continuum — Kaggle

Runtime **efímero** + workstation **persistente** (Kaggle Dataset).

### Secrets (Settings → Secrets)

- `NVIDIA_API_KEY` (opcional — sin él arranca igual, sin modelos NVIDIA)
- `OPENCODE_CLOUD_DATASET` = `usuario/dataset-privado`
- `OPENCODE_SERVER_PASSWORD` (opcional, basic auth de OpenCode)
- `CLOUDFLARE_TUNNEL_TOKEN` (opcional — túnel nombrado permanente)
- `GITHUB_TOKEN` / `GITHUB_REPO` (opcional, solo versionado de código)

### Acceso desde el teléfono

1. **Primario:** Cloudflare Tunnel (quick o named)
2. **Fallback:** Kaggle Jupyter Proxy (solo HTTP; la UI SPA puede fallar)

### Flujo

1. Bootstrap restaura el Dataset (o crea workstation nueva)
2. Arranca OpenCode Web en el puerto local (default 4096)
3. Intenta Cloudflare Tunnel; si falla, usa el proxy de Kaggle
4. Si `web_access.available` es true, abrí la URL en el teléfono
5. La última celda (keep-alive) mantiene la sesión ejecutando;
   si OpenCode no se recupera, la celda termina y el guardián
   externo relanza el kernel (ver `scripts/guardian/README.md`)

La URL es **temporal**. No la guardes en el Dataset.


In [ ]:
!pip install -q kagglehub
!pip install -q git+https://github.com/enrrutador/opencode-continuum.git@main


In [ ]:
import opencode_kaggle.bootstrap as bs

info = bs.bootstrap()
print("status:", info.get("status"))
print("recovery:", info.get("recovery"))
wa = info.get("web_access") or {}
print("provider:", wa.get("provider"))
print("opencode_listening:", wa.get("opencode_listening"))
print("access status:", wa.get("status"))
print("url_redacted:", wa.get("url_redacted"))
if wa.get("available") and wa.get("url"):
    print("\nAbrí esta URL en el teléfono (sesión actual):")
    print(wa["url"])
elif wa.get("available") and wa.get("status") == "named_tunnel_running":
    print("\nNamed Tunnel activo — usá el hostname configurado en Cloudflare.")
else:
    print("\nAcceso externo no validado.")
    print("message:", wa.get("message"))


In [ ]:
# Keep-alive: mantiene esta celda ejecutando para que Kaggle no mate
# la sesión por inactividad. Si OpenCode deja de responder y el watchdog
# interno no lo revive en ~5 min, la celda termina: el guardián externo
# (GitHub Actions / scripts/guardian) relanza el kernel y todo se restaura.
import socket
import time
from datetime import datetime, timezone

from opencode_cloud.ports import get_opencode_port

PORT = get_opencode_port()
TICK_S = 60
GRACE_TICKS = 5
BEAT_TICKS = 10

misses = 0
ticks = 0
while True:
    s = socket.socket()
    s.settimeout(5)
    try:
        s.connect(("127.0.0.1", PORT))
        alive = True
    except OSError:
        alive = False
    finally:
        s.close()
    ticks += 1
    now = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M UTC")
    if alive:
        misses = 0
        if ticks % BEAT_TICKS == 1:
            print(f"[keep-alive] {now} OpenCode responde en {PORT}", flush=True)
    else:
        misses += 1
        print(f"[keep-alive] {now} puerto {PORT} sin respuesta ({misses}/{GRACE_TICKS})", flush=True)
        if misses > GRACE_TICKS:
            print("[keep-alive] sin recuperación: termino para que el guardián relance el kernel.", flush=True)
            break
    time.sleep(TICK_S)
